# Bayesian Persuasion for Infrastructure — Formal Model

The planner here cannot build roads or levy tolls; it can only **control information**
about uncertain network conditions (incidents, hazards, the congestion state). Travelers
self-route to minimize their own travel time. The planner *commits* to a signaling scheme
and, through the routing it induces, optimizes a social objective.

This notebook fixes the notation and states the model the `bp` package implements.
Symbols are tied to code objects throughout (e.g. a state $\omega$ is a `Scenario`).

**Modeling stance (current).** Travelers are *homogeneous* and have *no preference model*
over the various metrics: each cares only about its own realized travel time. Every other
metric (emissions, hazard, monetary cost, policing) enters only the **planner's** objective.
The metric trade-off therefore lives entirely in planner weights, not in receiver utilities.

## 1. Primitives

| Object | Symbol | Code |
|---|---|---|
| Directed network | $G=(V,A)$ | `InfrastructureGraph` |
| Instrumented arcs (planner can sense/signal) | $I \subseteq A$ | `InfrastructureGraph.I` |
| Population of travelers | $N=\{1,\dots,n\}$ | `World.individuals` |
| Demand of traveler $i$ | $(o_i,d_i)$ | `Individual.demand` |
| Admissible routes of $i$ (simple $o_i\!\to\! d_i$ paths) | $\mathcal{R}_i \subseteq 2^A$ | (routing layer, TBD) |
| State of the world | $\omega \in \Omega$ | `Scenario` |
| Common prior over states | $\mu \in \Delta(\Omega)$ | `Prior` / `FinitePrior` |

A route profile is $r=(r_i)_{i\in N}$ with $r_i\in\mathcal{R}_i$; it induces arc flows
$$ x_a(r) \;=\; \sum_{i\in N} \mathbb{1}\!\left[a \in r_i\right], \qquad a\in A. $$

## 2. States, prior, and congestion

A **state** $\omega\in\Omega$ fixes every uncertain quantity at once. In code $\omega$ is a
`Scenario`, assigning to each arc $a$ a free-flow travel time $t_a(\omega)$, hazard
$h_a(\omega)$, monetary cost $c_a(\omega)$, an emission factor, and to each node $v$ a
policing level $p_v(\omega)$. The planner's uncertainty is the prior $\mu\in\Delta(\Omega)$,
from which states are drawn ($\texttt{Prior.sample}$).

**Congestion (BPR).** Realized arc travel time depends on the state *and* on the aggregate
flow — this is what couples the travelers:
$$ \tau_a(\omega, x) \;=\; t_a(\omega)\,\Big(1 + \alpha\big(x_a / c_a\big)^{\beta}\Big), $$
with capacity $c_a$ and parameters $\alpha,\beta$. This is exactly
`InfrastructureGraph.actual_travel_times`. Emissions $e_a(\omega,x)$ scale the arc distance
by the realized slowdown $\tau_a(\omega,x)/t_a(\omega)$ (`actual_emissions`).

## 3. Receiver utility

Traveler $i$ cares only about its own realized travel time. Under state $\omega$ and route
profile $r$, the cost of taking route $r_i$ is
$$ C_i\big(r_i;\, \omega,\, r_{-i}\big) \;=\; \sum_{a \in r_i} \tau_a\big(\omega,\, x(r)\big). $$

There are **no per-metric preference weights** at the receiver level: travelers are identical
self-routers. Heterogeneous preferences (weights, or a partial order over metrics) are a
deliberate non-goal for now and can be reintroduced later by making $C_i$ traveler-specific.

## 4. Sender (planner) objective

The planner evaluates a **social cost** on the realized state and flows. With non-negative
planner weights $\theta = (\theta_\tau, \theta_e, \theta_h, \theta_c, \theta_p)$,
$$
\Psi(\omega, x) \;=\; \sum_{a\in A} x_a\Big[\, \theta_\tau\,\tau_a(\omega,x)
   \;+\; \theta_e\, e_a(\omega,x) \;+\; \theta_h\, h_a(\omega) \;+\; \theta_c\, c_a(\omega) \,\Big]
   \;+\; \theta_p \sum_{v\in V} p_v(\omega)\, \rho_v(x),
$$
where $\rho_v(x)$ is the throughput/exposure at node $v$. Special cases: total travel time
($\theta=e_\tau$), total emissions, hazard exposure, or any weighted blend. The planner is
**not** a traveler — the entire metric trade-off is encoded in $\theta$, chosen by the
analyst. The planner minimizes the expected social cost $\mathbb{E}\,[\Psi(\omega,x)]$.

## 5. Signal space and timing (commitment)

The planner commits to a **signaling scheme** $\pi:\Omega \to \Delta(S)$ over a signal space
$S$. The Bayesian-persuasion timing is:

1. The planner publicly commits to $\pi$.
2. Nature draws $\omega \sim \mu$; a signal $s \sim \pi(\cdot \mid \omega)$ is emitted.
3. Each traveler forms the posterior $\mu(\cdot \mid s) \propto \pi(s \mid \omega)\,\mu(\omega)$
   and chooses a route.
4. Flows realize, costs realize, and the planner incurs $\Psi(\omega, x)$.

**Public vs. private.** Under *public* signaling a single $s$ is broadcast to everyone. Under
*private* signaling $s=(s_i)_{i\in N}$ and traveler $i$ observes only $s_i$ — the richer regime
for steering a congested population. Both are instances of the scheme $\pi$ above.

## 6. Equilibrium

Fix a scheme $\pi$. The travelers play a **Bayes–Nash equilibrium** of the induced
incomplete-information routing game: strategies $\sigma_i : S_i \to \Delta(\mathcal{R}_i)$ such
that, for every $i$, every signal $s_i$, and every route $r_i$ in the support of
$\sigma_i(s_i)$,
$$ r_i \;\in\; \arg\min_{r_i' \in \mathcal{R}_i}\;
   \mathbb{E}\big[\, C_i(r_i';\, \omega,\, r_{-i}) \,\big|\, s_i \,\big], $$
the expectation taken over the posterior on $\omega$ and the others' equilibrium routes
$r_{-i}$. The planner chooses $\pi$ (and, when equilibria are not unique, a selection rule) to
minimize $\mathbb{E}[\Psi(\omega, x)]$.

## 7. Tractable form: recommendations + obedience (BCE)

By the **revelation principle** for information design, it is without loss to restrict to
*direct* schemes whose signal to traveler $i$ is a route **recommendation** $r_i\in\mathcal{R}_i$.
A scheme is then a joint law over states and recommendation profiles,
$q \in \Delta\big(\Omega \times \prod_i \mathcal{R}_i\big)$, whose state-marginal is the prior.
It is implementable iff it is **obedient** — each traveler weakly prefers the recommended route:
$$ \sum_{\omega,\, r_{-i}} q(\omega, r_i, r_{-i})\,
   \Big[\, C_i(r_i;\omega,r) - C_i\big(r_i';\omega,(r_i',r_{-i})\big) \,\Big] \;\le\; 0
   \qquad \forall\, i,\; r_i,\; r_i' \in \mathcal{R}_i. $$
The planner then solves
$$
\min_{q \ge 0}\; \sum_{\omega,\, r} q(\omega, r)\,\Psi\big(\omega, x(r)\big)
\quad\text{s.t.}\quad
\sum_{r} q(\omega, r) = \mu(\omega)\;\;\forall \omega,\quad \text{and obedience.}
$$

**The congestion caveat.** With $\Omega$ and the route sets finite this is *linear in $q$
except* that $C_i$ and $\Psi$ depend on the profile through the BPR flow $x(r)$ — congestion
couples travelers and makes the coefficients flow-dependent. Two standard resolutions, and the
key fork to resolve before the optimization layer:

- **Nonatomic limit.** Replace $N$ by a continuum of mass; work with route-flow distributions.
  Obedience and objective become integrals over flows and the program is convex (LP / conic) in
  the flow measure. *(Information design in nonatomic routing games.)*
- **Atomic.** Keep finitely many travelers; the program is finite but generally nonconvex — a
  job for the `gurobipy` layer.

**Sanity checks.** If $\alpha=0$ (no congestion), $\tau_a=t_a(\omega)$, costs decouple across
travelers, and the problem reduces to per-traveler single-agent persuasion (classic
Kamenica–Gentzkow) with an exact obedience LP. *Full information* ($\pi$ reveals $\omega$) and
*no information* ($\pi$ constant) are always feasible and bracket the planner's achievable value.

## 8. Symbol → code map, and what is still missing

| Math | Meaning | Code today |
|---|---|---|
| $G=(V,A)$, $I$ | network, instrumented arcs | `InfrastructureGraph` |
| $\omega$, $\mu$ | state, prior | `Scenario`, `Prior`/`FinitePrior`/`SampledPrior` |
| $t_a,h_a,c_a,p_v$ | per-arc / per-node metrics | `Scenario` fields, `InfrastructureGraph` props |
| $\tau_a(\omega,x)$, $e_a(\omega,x)$ | congested travel time, emissions | `actual_travel_times`, `actual_emissions` |
| $x(r)$ | arc flows from routes | `World.realize(routes)` |
| $\mathcal{R}_i$ | route sets | **missing** — routing layer |
| $C_i$, best response | receiver behavior | **missing** — receiver layer |
| $\Psi$, $\theta$ | planner social cost | **missing** — objective |
| $\pi$ / $q$, obedience | signaling / information design | **missing** — sender + solver |

**Next code steps implied by this model:** (1) enumerate/serve route sets $\mathcal{R}_i$ and
receiver best response $C_i$; (2) the planner objective $\Psi(\omega,x)$; (3) the obedience
program over $q$ (nonatomic LP first). Decide the **atomic vs. nonatomic** fork before (3).

## Grounding in code

A quick check that the primitives above are real objects: build a reproducible random world,
read off $G$, $I$, the state $\omega$, and the congestion map $\tau$.

In [1]:
from bp.world import random_grid_world, Scenario
import networkx as nx

# A reproducible 3x3 grid (states/metrics fixed by the seed).
world = random_grid_world(
    rows=3,
    cols=3,
    demands={((0, 0), (2, 2)): 5, ((2, 2), (0, 0)): 2},
    seed=0,
    instrumented_fraction=0.3,
)
print(f"|N| = {world.total_population} travelers, |V| = {len(world.V)}, |A| = {len(world.A)}")
print("instrumented arcs I:", sorted(world.I, key=str))

|N| = 7 travelers, |V| = 9, |A| = 24
instrumented arcs I: [((0, 0), (1, 0)), ((1, 0), (1, 1)), ((1, 0), (2, 0)), ((1, 2), (2, 2)), ((2, 0), (2, 1)), ((2, 1), (1, 1)), ((2, 1), (2, 0)), ((2, 2), (2, 1))]


In [2]:
# A state omega (the nominal Scenario) and the BPR congestion map tau(omega, x):
# route everyone on a shortest free-flow path, then realize the congested state.
omega = Scenario.from_world("nominal", world)

G = world.network.graph
routes = {
    ind: list(zip(p[:-1], p[1:]))
    for ind in world.individuals
    for p in [nx.shortest_path(G, ind.demand.origin, ind.demand.destination, weight="travel_time")]
}
realized = world.realize(routes, name="rush_hour")

congested = sum(realized.travel_time[a] > omega.travel_time[a] + 1e-9 for a in world.A)
print(f"arcs whose travel time rose under congestion: {congested}/{len(world.A)}")

arcs whose travel time rose under congestion: 8/24
